# SALI-PyCCE Colab Training Workflow

This notebook trains the SALI-style model on analytic noisy CPMG traces, displays the required diagnostic figures, and leaves a path for PyCCE benchmark generation.

## 1. Environment and GPU Check

In [ ]:
!nvidia-smi || true

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Repository Setup and Install

In [ ]:
from pathlib import Path

# If you opened this notebook outside the repository, clone or upload the repo first.
# !git clone https://github.com/userwhe/sali-pycce-prototype.git
# %cd sali-pycce-prototype

!python -m pip install -q -e .[dev]
Path('runs').mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)

## 3. Synthetic Research Sample

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display

from sali_pycce.config import RESEARCH_CONFIG
from sali_pycce.data import SyntheticSALIDataset
from sali_pycce.visualize import plot_raw_traces_and_spins

sim = RESEARCH_CONFIG.build_simulator(shots=RESEARCH_CONFIG.shots)
spec = RESEARCH_CONFIG.build_heatmap_spec()
sample_ds = SyntheticSALIDataset(
    1,
    simulator=sim,
    heatmap_spec=spec,
    min_spins=RESEARCH_CONFIG.min_spins,
    max_spins=RESEARCH_CONFIG.max_spins,
    az_range=RESEARCH_CONFIG.az_range,
    aperp_range=RESEARCH_CONFIG.aperp_range,
    seed=2026,
    noisy=True,
)
item = sample_ds[0]
plot_raw_traces_and_spins(
    item['taus_us'].numpy(),
    item['signals'].numpy(),
    item['spins'].numpy(),
    spec,
    'runs/raw_traces_and_spins.png',
)

fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
ax.imshow(item['heatmap'][0].numpy(), origin='lower', aspect='auto')
ax.set_title('Ground-truth heatmap label')
ax.set_xlabel('A_z pixel')
ax.set_ylabel('A_perp pixel')
fig.savefig('runs/ground_truth_heatmap.png', dpi=160)
plt.close(fig)

display(Image('runs/raw_traces_and_spins.png'))
display(Image('runs/ground_truth_heatmap.png'))

## 4. Colab Smoke Training Command

In [ ]:
!python scripts/colab_train.py \
  --preset smoke \
  --epochs 1 \
  --batch-size 16 \
  --checkpoint checkpoints/colab_smoke.pt \
  --history runs/colab_smoke_history.csv \
  --metrics runs/colab_smoke_metrics.json

## 5. Colab Medium or Research Training Command

In [ ]:
!python scripts/colab_train.py \
  --preset colab-medium \
  --epochs 20 \
  --batch-size 128 \
  --checkpoint checkpoints/colab_medium.pt \
  --history runs/colab_medium_history.csv \
  --metrics runs/colab_medium_metrics.json

# First full research run:
# !python scripts/colab_train.py --preset research --epochs 30 --batch-size 128 \
#   --checkpoint checkpoints/research.pt --history runs/research_history.csv \
#   --metrics runs/research_metrics.json

## 6. Train and Validation Loss Curve

In [ ]:
from pathlib import Path
from IPython.display import Image, display

from sali_pycce.visualize import plot_loss_history

history_csv = Path('runs/colab_medium_history.csv')
if not history_csv.exists():
    history_csv = Path('runs/colab_smoke_history.csv')
plot_loss_history(history_csv, 'runs/loss_curve.png')
display(Image('runs/loss_curve.png'))

## 7. Target Heatmap Vs Predicted Heatmap

In [ ]:
import torch

from sali_pycce.data import SyntheticSALIDataset
from sali_pycce.heatmap import HeatmapSpec
from sali_pycce.model import SALINet
from sali_pycce.physics import AnalyticCPMGSimulator
from sali_pycce.visualize import plot_heatmap_comparison

checkpoint = Path('checkpoints/colab_medium.pt')
if not checkpoint.exists():
    checkpoint = Path('checkpoints/colab_smoke.pt')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(checkpoint, map_location=device)
ckpt_args = ckpt.get('args', {})
spec = HeatmapSpec(**ckpt.get('heatmap_spec', {}))
model = SALINet(output_shape=(spec.height, spec.width)).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

tau_range = (float(ckpt_args.get('tau_start_us', 0.0)), float(ckpt_args.get('tau_stop_us', 40.0)))
sim = AnalyticCPMGSimulator(
    b_gauss=float(ckpt_args.get('b_gauss', 525.0)),
    pulses=(32, 256),
    tau_ranges_us=(tau_range, tau_range),
    signal_points=int(ckpt_args.get('signal_points', 4000)),
    shots=ckpt_args.get('shots', 1000),
    t2_us=ckpt_args.get('t2_us', 800.0),
    t2_stretch=float(ckpt_args.get('t2_stretch', 1.0)),
)
pred_ds = SyntheticSALIDataset(
    1,
    simulator=sim,
    heatmap_spec=spec,
    min_spins=int(ckpt_args.get('min_spins', 1)),
    max_spins=int(ckpt_args.get('max_spins', 20)),
    az_range=spec.az_range,
    aperp_range=spec.aperp_range,
    seed=777,
)
pred_item = pred_ds[0]
with torch.no_grad():
    pred = model(pred_item['signals'][None].to(device)).cpu()[0, 0].numpy()
plot_heatmap_comparison(pred_item['heatmap'][0].numpy(), pred, 'runs/heatmap_comparison.png')
display(Image('runs/heatmap_comparison.png'))

## 8. Precision, Recall, and MAE

In [ ]:
from sali_pycce.visualize import plot_metric_history

plot_metric_history(history_csv, 'runs/metric_curve.png')
display(Image('runs/metric_curve.png'))

## 9. Save Checkpoints and Runs to Drive

In [ ]:
# Optional Drive backup in Colab.
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/sali-pycce-runs
# !cp -r checkpoints runs /content/drive/MyDrive/sali-pycce-runs/